# Step 3E.0 - Hybrid Gate Replay

This notebook is a **dev-only, gate-only replay** experiment for `counterfact_direction1_v1`.

It does not run LLaDA. It composes existing subject-gated outputs with base outputs:

```text
if replay_gate(prompt, edit) == ON:
    use existing subject-gated method output
else:
    use base output
```

Goal: test whether an edit-intent gate can keep rewrite/paraphrase activation high while shutting off same-subject target over-injection.

This notebook uses only `dev_tune_200` artifacts and must not touch `analysis_500`, `final_test_500`, or `final_test_full`.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Environment And Paths


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import platform
import re
import string
import subprocess
import sys
import unicodedata
from collections import Counter, defaultdict

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
RUN_ROOT = PROJECT_DIR / "runs/counterfact_direction1_v1"
OUT_REPORT = RUN_ROOT / "dev_tune_200_hybrid_gate_replay_v1"

PROTOCOL_VERSION = "counterfact_direction1_v1"
ALLOW_OVERWRITE = False

print("python:", sys.version)
print("platform:", platform.platform())
print("project dir:", PROJECT_DIR)
print("protocol_version:", PROTOCOL_VERSION)

try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
except Exception as exc:
    commit = None
    print("git commit unavailable:", repr(exc))
print("git commit:", commit)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from llada_experiment_reports import harmonic_mean, paired_bootstrap_delta_by_case


## Preflight

This cell checks that Step 3C.1, Step 3D, and Step 3D.1 artifacts exist before building any replay table.


In [ ]:
PROTOCOL_DEV = RUN_ROOT / "protocol/dev_tune_200.jsonl"
THRESHOLDS = RUN_ROOT / "dev_tune_200_runtime_baseline_thresholds.json"
DENSE_REPORT = RUN_ROOT / "dev_tune_200_dense_pareto_subject_v1"
STEP3D_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_report_v1"
STEP3D1_REPORT = RUN_ROOT / "dev_tune_200_same_subject_stress_midkl_v1"
STRESS_INPUT = RUN_ROOT / "same_subject_stress_inputs/dev_tune_200_same_subject_stress.jsonl"
STRESS_SUMMARY = RUN_ROOT / "same_subject_stress_inputs/summary.json"

required_paths = [
    PROTOCOL_DEV,
    THRESHOLDS,
    DENSE_REPORT / "dense_pareto_summary.csv",
    DENSE_REPORT / "best_by_kl_budget.csv",
    DENSE_REPORT / "best_guided_by_kl_budget.csv",
    STEP3D_REPORT / "report_summary.json",
    STEP3D1_REPORT / "report_summary.json",
    STRESS_INPUT,
    STRESS_SUMMARY,
]
for path in required_paths:
    assert path.exists(), f"Missing required artifact: {path}"

for path in [STEP3D_REPORT / "report_summary.json", STEP3D1_REPORT / "report_summary.json"]:
    with path.open() as f:
        summary = json.load(f)
    assert summary.get("analysis_500_used") is False, f"analysis_500 used in {path}"
    assert summary.get("final_test_used") is False, f"final_test used in {path}"

if OUT_REPORT.exists() and not ALLOW_OVERWRITE:
    raise RuntimeError(
        f"Output report already exists: {OUT_REPORT}. "
        "Delete it manually, choose a new report name, or set ALLOW_OVERWRITE=True."
    )
OUT_REPORT.mkdir(parents=True, exist_ok=True)

print("Preflight OK.")
print("Output report:", OUT_REPORT)


## Replay Registry

The source labels are existing subject-gated runs. The replay labels are the hypothetical hybrid-gated methods that would be decoded later only if replay looks promising.


In [ ]:
BASE_NORMAL = {
    "label": "base",
    "method": "base",
    "run_dir": "dev_tune_200_base_seed0",
}
BASE_STRESS = {
    "label": "base",
    "method": "base",
    "run_dir": "dev_tune_200_same_subject_stress_base",
}

METHOD_SPECS = [
    {
        "source_label": "prompt_memory_gated_subject_gs1.0",
        "replay_method_label": "prompt_memory_gated_hybrid",
        "method": "prompt_memory_gated_subject",
        "family": "prompt_memory",
        "guidance_scale": 1.0,
        "normal_run_dir": "dev_tune_200_gated_prompt_memory_subject",
        "stress_run_dir": "dev_tune_200_same_subject_stress_prompt_memory_subject",
    },
    {
        "source_label": "myopic_score_gated_subject_gs1.5",
        "replay_method_label": "myopic_score_gated_hybrid_gs1.5",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 1.5,
        "normal_run_dir": "dev_tune_200_dense_pareto_bridge_controls_subject_gs150",
        "stress_run_dir": "dev_tune_200_same_subject_stress_myopic_gs150",
    },
    {
        "source_label": "myopic_score_gated_subject_gs1.75",
        "replay_method_label": "myopic_score_gated_hybrid_gs1.75",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 1.75,
        "normal_run_dir": "dev_tune_200_dense_pareto_bridge_controls_subject_gs175",
        "stress_run_dir": "dev_tune_200_same_subject_stress_myopic_gs175",
    },
    {
        "source_label": "myopic_score_gated_subject_gs2.0",
        "replay_method_label": "myopic_score_gated_hybrid_gs2.0",
        "method": "myopic_score_gated_subject",
        "family": "myopic_score",
        "guidance_scale": 2.0,
        "normal_run_dir": "dev_tune_200_sweep_bridge_controls_subject_gs200",
        "stress_run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
    },
    {
        "source_label": "mc_bridge_gated_subject_gs1.5",
        "replay_method_label": "mc_bridge_gated_hybrid_gs1.5",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 1.5,
        "normal_run_dir": "dev_tune_200_dense_pareto_mc_bridge_subject_gs150",
        "stress_run_dir": "dev_tune_200_same_subject_stress_mc_bridge_gs150",
    },
    {
        "source_label": "mc_bridge_gated_subject_gs1.75",
        "replay_method_label": "mc_bridge_gated_hybrid_gs1.75",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 1.75,
        "normal_run_dir": "dev_tune_200_dense_pareto_mc_bridge_subject_gs175",
        "stress_run_dir": "dev_tune_200_same_subject_stress_mc_bridge_gs175",
    },
    {
        "source_label": "mc_bridge_gated_subject_gs2.0",
        "replay_method_label": "mc_bridge_gated_hybrid_gs2.0",
        "method": "mc_bridge_gated_subject",
        "family": "mc_bridge",
        "guidance_scale": 2.0,
        "normal_run_dir": "dev_tune_200_sweep_mc_bridge_subject_gs200",
        "stress_run_dir": "dev_tune_200_same_subject_stress_mc_bridge_subject_gs200",
    },
    {
        "source_label": "no_rollout_bridge_gated_subject_gs1.5",
        "replay_method_label": "no_rollout_bridge_gated_hybrid_gs1.5",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 1.5,
        "normal_run_dir": "dev_tune_200_dense_pareto_bridge_controls_subject_gs150",
        "stress_run_dir": "dev_tune_200_same_subject_stress_no_rollout_gs150",
    },
    {
        "source_label": "no_rollout_bridge_gated_subject_gs1.75",
        "replay_method_label": "no_rollout_bridge_gated_hybrid_gs1.75",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 1.75,
        "normal_run_dir": "dev_tune_200_dense_pareto_bridge_controls_subject_gs175",
        "stress_run_dir": "dev_tune_200_same_subject_stress_no_rollout_gs175",
    },
    {
        "source_label": "no_rollout_bridge_gated_subject_gs2.0",
        "replay_method_label": "no_rollout_bridge_gated_hybrid_gs2.0",
        "method": "no_rollout_bridge_gated_subject",
        "family": "no_rollout_bridge",
        "guidance_scale": 2.0,
        "normal_run_dir": "dev_tune_200_sweep_bridge_controls_subject_gs200",
        "stress_run_dir": "dev_tune_200_same_subject_stress_bridge_controls_subject_gs200",
    },
]

for run_name in {BASE_NORMAL["run_dir"], BASE_STRESS["run_dir"]} | {spec["normal_run_dir"] for spec in METHOD_SPECS} | {spec["stress_run_dir"] for spec in METHOD_SPECS}:
    run_dir = RUN_ROOT / run_name
    for artifact in ["run_config.json", "summary.json", "per_case_results.jsonl"]:
        assert (run_dir / artifact).exists(), f"Missing {artifact}: {run_dir}"

print("Replay method specs:")
print(json.dumps(METHOD_SPECS, indent=2))


## Utility Functions


In [ ]:
def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def mean_or_none(values):
    vals = [float(v) for v in values if v is not None]
    return sum(vals) / len(vals) if vals else None


def first_output(row):
    outputs = row.get("sample_outputs") or []
    return outputs[0] if outputs else row.get("greedy_output", "")


def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text or "")).lower()
    text = text.replace("[mask]", " ").replace("<mask>", " ")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def remove_phrase(text, phrase):
    text_norm = normalize_text(text)
    phrase_norm = normalize_text(phrase)
    if phrase_norm:
        text_norm = re.sub(r"\b" + re.escape(phrase_norm) + r"\b", " ", text_norm)
    return re.sub(r"\s+", " ", text_norm).strip()


# Keep relation-bearing words such as works, born, located, citizenship, language, and capital.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "have",
    "he", "her", "his", "in", "is", "it", "its", "of", "on", "or", "she", "that",
    "the", "their", "they", "to", "was", "were", "what", "where", "which", "who",
    "whom", "whose", "with", "did", "does", "do",
}


def content_text(prompt, *, subject="", target_new="", target_true=""):
    text = normalize_text(prompt)
    for phrase in [subject, target_new, target_true, "mask"]:
        text = remove_phrase(text, phrase)
    tokens = [tok for tok in text.split() if tok not in STOPWORDS and len(tok) > 1]
    return " ".join(tokens)


def char_ngrams(text, n=3):
    text = f"  {normalize_text(text)}  "
    if len(text) < n:
        return Counter([text]) if text.strip() else Counter()
    return Counter(text[i:i+n] for i in range(len(text) - n + 1))


def cosine_counter(left, right):
    if not left or not right:
        return 0.0
    dot = sum(left[k] * right.get(k, 0) for k in left)
    left_norm = math.sqrt(sum(v * v for v in left.values()))
    right_norm = math.sqrt(sum(v * v for v in right.values()))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    return dot / (left_norm * right_norm)


def word_jaccard(left, right):
    a = set(normalize_text(left).split())
    b = set(normalize_text(right).split())
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def relation_similarity(left, right):
    char_sim = cosine_counter(char_ngrams(left), char_ngrams(right))
    word_sim = word_jaccard(left, right)
    return max(char_sim, word_sim)


def render_template(template, subject):
    template = str(template or "")
    if "{}" in template:
        return template.format(subject)
    if "{subject}" in template:
        return template.replace("{subject}", subject)
    return template


def stress_bucket_from_prompt_id(prompt_id):
    prompt_id = str(prompt_id or "")
    if "_same_subject_template_" in prompt_id:
        return "same_subject_template"
    if "_generation_" in prompt_id:
        return "generation"
    if "_attribute_" in prompt_id:
        return "attribute"
    return "unknown"


def row_bucket(row, source):
    if source == "stress":
        return stress_bucket_from_prompt_id(row.get("prompt_id"))
    return str(row.get("bucket"))


def prompt_key(row, source):
    bucket = row_bucket(row, source)
    edit_id = str(row.get("edit_id") or row.get("case_id"))
    prompt_id = row.get("prompt_id") or row.get("prompt_uid")
    if prompt_id is None:
        prompt = str(row.get("prompt") or row.get("prompt_text") or row.get("rendered_prompt") or "")
        prompt_id = "prompt_sha1:" + hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:16]
    return (edit_id, bucket, str(prompt_id))


def load_rows_for_method(run_name, method, *, source):
    path = RUN_ROOT / run_name / "per_case_results.jsonl"
    rows = []
    for row in read_jsonl(path):
        row_method = row.get("method_variant") or row.get("method")
        raw_method = row.get("method")
        if method == "base":
            # Older base artifacts may carry a non-canonical method_variant while preserving method=base.
            if row_method != "base" and raw_method != "base":
                continue
        elif row_method != method:
            continue
        bucket = row_bucket(row, source)
        if bucket == "unknown":
            continue
        rows.append({**row, "replay_bucket": bucket})
    return rows


def index_rows(rows, source):
    indexed = {}
    duplicates = Counter()
    for row in rows:
        key = prompt_key(row, source)
        if key in indexed:
            duplicates[key] += 1
            key = (*key, f"duplicate:{duplicates[key]}")
        indexed[key] = row
    assert len(indexed) == len(rows), "Prompt-row indexing collapsed rows"
    return indexed


## Load Protocol Metadata And Relation Prototypes

The relation-bank prototype uses only `dev_tune_200` rewrite templates. It does **not** use evaluation paraphrase prompts as gate templates.


In [ ]:
def target_new_text(record):
    if isinstance(record.get("target_new"), dict):
        return str(record["target_new"].get("text") or record["target_new"].get("str") or record.get("target") or "")
    return str(record.get("target_new") or record.get("target") or "")


def target_true_text(record):
    if isinstance(record.get("target_true"), dict):
        return str(record["target_true"].get("text") or record["target_true"].get("str") or record.get("old_target") or "")
    rewrite = record.get("requested_rewrite") or {}
    if isinstance(rewrite, dict) and isinstance(rewrite.get("target_true"), dict):
        return str(rewrite["target_true"].get("str") or record.get("old_target") or "")
    return str(record.get("target_true") or record.get("old_target") or "")


def record_subject(record):
    rewrite = record.get("requested_rewrite") or {}
    if isinstance(rewrite, dict) and rewrite.get("subject"):
        return str(rewrite["subject"])
    return str(record.get("subject") or "")


def record_template(record):
    rewrite = record.get("requested_rewrite") or {}
    if isinstance(rewrite, dict) and rewrite.get("prompt"):
        return str(rewrite["prompt"])
    return str(record.get("rewrite_template") or record.get("prompt") or "")


manifest_rows = list(read_jsonl(PROTOCOL_DEV))
assert len(manifest_rows) == 200, f"Expected 200 dev_tune rows, got {len(manifest_rows)}"

meta_by_edit = {}
relation_templates = defaultdict(list)
for record in manifest_rows:
    edit_id = str(record.get("id") or record.get("edit_id") or record.get("case_id"))
    subject = record_subject(record)
    relation_id = str(record.get("relation_id") or "")
    rewrite_template = record_template(record)
    target_new = target_new_text(record)
    target_true = target_true_text(record)
    rendered = render_template(rewrite_template, subject)
    relation_text = content_text(rendered, subject=subject, target_new=target_new, target_true=target_true)
    meta = {
        "edit_id": edit_id,
        "case_id": str(record.get("case_id") or edit_id),
        "subject": subject,
        "relation_id": relation_id,
        "rewrite_template": rewrite_template,
        "rewrite_relation_text": relation_text,
        "target_new": target_new,
        "target_true": target_true,
        "target_length_bin": str(record.get("target_length_bin") or ""),
    }
    meta_by_edit[edit_id] = meta
    if relation_id and relation_text:
        relation_templates[relation_id].append(relation_text)

relation_bank = {
    relation_id: sorted(set(texts))
    for relation_id, texts in relation_templates.items()
}

print("dev edits:", len(meta_by_edit))
print("relation ids:", len(relation_bank))
print("example relation prototype counts:", dict(list((k, len(v)) for k, v in relation_bank.items())[:5]))


## Build Row-Level Gate Replay Dataset

Each dataset row aligns one base prompt-row with one existing subject-gated method prompt-row. Gate features are computed from the prompt and edit metadata, not from evaluation paraphrase templates.


In [ ]:
base_normal_rows = load_rows_for_method(BASE_NORMAL["run_dir"], BASE_NORMAL["method"], source="normal")
base_stress_rows = load_rows_for_method(BASE_STRESS["run_dir"], BASE_STRESS["method"], source="stress")

positive_buckets = {"rewrite", "declarative_paraphrases", "qa_format_generalization"}
negative_buckets = {"near_locality", "far_locality", "same_subject_template", "generation"}
main_positive_buckets = {"rewrite", "declarative_paraphrases"}


def metric(row, key):
    value = row.get(key)
    return None if value is None else float(value)


def row_prompt(row):
    return str(row.get("prompt") or row.get("prompt_text") or row.get("rendered_prompt") or "")


def prompt_digest(row):
    prompt = row_prompt(row)
    if not prompt:
        return None
    return hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:16]


def normalized_prompt_digest(row):
    prompt = normalize_text(row_prompt(row))
    if not prompt:
        return None
    return hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:16]


def edit_keys_for_row(row):
    keys = []
    for key in [row.get("edit_id"), row.get("case_id")]:
        if key is not None:
            key = str(key)
            if key and key not in keys:
                keys.append(key)
    return keys


def prompt_ids_for_row(row):
    ids = []
    for key in [row.get("prompt_id"), row.get("prompt_uid")]:
        if key is not None:
            key = str(key)
            if key and key not in ids:
                ids.append(key)
    return ids


def build_base_lookup(rows, source):
    lookup = {}
    occurrence_counts = defaultdict(int)
    for row in rows:
        bucket = row_bucket(row, source)
        edit_ids = edit_keys_for_row(row)
        prompt_ids = prompt_ids_for_row(row)
        digest = prompt_digest(row)
        norm_digest = normalized_prompt_digest(row)
        primary_edit = edit_ids[0] if edit_ids else str(row.get("edit_id") or row.get("case_id") or "")
        occurrence_key = (primary_edit, bucket)
        occurrence_index = occurrence_counts[occurrence_key]
        occurrence_counts[occurrence_key] += 1

        for edit_id in edit_ids:
            for prompt_id in prompt_ids:
                lookup.setdefault(("prompt_id", edit_id, bucket, prompt_id), row)
            if digest:
                lookup.setdefault(("prompt_sha1", edit_id, bucket, digest), row)
            if norm_digest:
                lookup.setdefault(("prompt_norm_sha1", edit_id, bucket, norm_digest), row)
            lookup.setdefault(("occurrence", edit_id, bucket, occurrence_index), row)
    return lookup


base_lookup = {}
base_lookup.update(build_base_lookup(base_normal_rows, "normal"))
base_lookup.update(build_base_lookup(base_stress_rows, "stress"))
base_alignment_counts = Counter()
base_missing_examples = []


def subject_match(prompt, subject):
    subject_norm = normalize_text(subject)
    prompt_norm = normalize_text(prompt)
    if not subject_norm:
        return False
    return re.search(r"\b" + re.escape(subject_norm) + r"\b", prompt_norm) is not None


def edit_advantage_for_row(base_row, method_row):
    # Only allow pre-decode/model-score features. Never fall back to exact_rate or TFPR.
    for key in ["guided_margin", "candidate_support_margin", "target_new_margin", "base_margin"]:
        for row_name, row in [("method", method_row), ("base", base_row)]:
            value = row.get(key)
            if value is not None:
                try:
                    return float(value), f"{row_name}.{key}"
                except (TypeError, ValueError):
                    pass
    return None, None


def inactive_gate_base_fallback(method_row):
    activation = method_row.get("gate_activation_rate")
    try:
        inactive = activation is not None and float(activation) == 0.0
    except (TypeError, ValueError):
        inactive = False
    if not inactive:
        return None
    # If the original subject gate was inactive, the subject-gated output is base-equivalent.
    row = dict(method_row)
    row["method"] = "base"
    row["method_variant"] = "base_replay_inactive_gate_fallback"
    row["sparse_guidance_kl"] = 0.0
    return row


def find_base_row(method_row, source, key, occurrence_index):
    edit_id, bucket, prompt_id = key[:3]
    edit_ids = edit_keys_for_row(method_row) or [str(edit_id)]
    prompt_ids = prompt_ids_for_row(method_row) or [str(prompt_id)]
    digest = prompt_digest(method_row)
    norm_digest = normalized_prompt_digest(method_row)
    candidates = []
    for candidate_edit_id in dict.fromkeys([str(edit_id), *edit_ids]):
        for candidate_prompt_id in dict.fromkeys([str(prompt_id), *prompt_ids]):
            candidates.append(("prompt_id", candidate_edit_id, bucket, candidate_prompt_id))
        if digest:
            candidates.append(("prompt_sha1", candidate_edit_id, bucket, digest))
        if norm_digest:
            candidates.append(("prompt_norm_sha1", candidate_edit_id, bucket, norm_digest))
        candidates.append(("occurrence", candidate_edit_id, bucket, occurrence_index))

    for candidate in candidates:
        if candidate in base_lookup:
            base_alignment_counts[candidate[0]] += 1
            return base_lookup[candidate], candidate[0]

    fallback = inactive_gate_base_fallback(method_row)
    if fallback is not None:
        base_alignment_counts["inactive_gate_fallback"] += 1
        return fallback, "inactive_gate_fallback"

    base_alignment_counts["missing"] += 1
    base_missing_examples.append((method_row.get("method_variant") or method_row.get("method"), key, row_prompt(method_row)[:120]))
    return None, None


def feature_row(base_row, method_row, spec, key, alignment_source):
    edit_id, bucket, prompt_id = key[:3]
    meta = meta_by_edit.get(edit_id, {})
    prompt = row_prompt(base_row) or row_prompt(method_row)
    subject = meta.get("subject") or str(base_row.get("subject") or method_row.get("subject") or "")
    target_new = meta.get("target_new") or ""
    target_true = meta.get("target_true") or ""
    prompt_relation = content_text(prompt, subject=subject, target_new=target_new, target_true=target_true)
    rewrite_relation = meta.get("rewrite_relation_text") or ""
    relation_id = meta.get("relation_id") or str(base_row.get("relation_id") or method_row.get("relation_id") or "")
    bank_sims = [relation_similarity(prompt_relation, proto) for proto in relation_bank.get(relation_id, [])]
    sim_bank = max(bank_sims) if bank_sims else 0.0
    sim_rewrite = relation_similarity(prompt_relation, rewrite_relation)
    adv, adv_source = edit_advantage_for_row(base_row, method_row)
    method_match = metric(method_row, "exact_rate") if bucket in positive_buckets else metric(method_row, "target_false_positive_rate")
    base_match = metric(base_row, "exact_rate") if bucket in positive_buckets else metric(base_row, "target_false_positive_rate")
    return {
        "method_source_label": spec["source_label"],
        "replay_method_label": spec["replay_method_label"],
        "family": spec["family"],
        "guidance_scale": spec["guidance_scale"],
        "edit_id": edit_id,
        "case_id": str(base_row.get("case_id") or method_row.get("case_id") or edit_id),
        "subject": subject,
        "relation_id": relation_id,
        "target_new": target_new,
        "target_true": target_true,
        "target_length_bin": meta.get("target_length_bin") or str(base_row.get("target_length_bin") or method_row.get("target_length_bin") or ""),
        "bucket": bucket,
        "prompt_id": prompt_id,
        "prompt": prompt,
        "base_output": first_output(base_row),
        "method_output": first_output(method_row),
        "base_target_new_match": base_match,
        "method_target_new_match": method_match,
        "base_exact_rate": metric(base_row, "exact_rate"),
        "method_exact_rate": metric(method_row, "exact_rate"),
        "base_token_f1": metric(base_row, "token_f1"),
        "method_token_f1": metric(method_row, "token_f1"),
        "base_malformed_rate": metric(base_row, "malformed_rate"),
        "method_malformed_rate": metric(method_row, "malformed_rate"),
        "base_target_false_positive_rate": metric(base_row, "target_false_positive_rate"),
        "method_target_false_positive_rate": metric(method_row, "target_false_positive_rate"),
        "method_sparse_guidance_kl": metric(method_row, "sparse_guidance_kl"),
        "subject_match": subject_match(prompt, subject),
        "relation_sim_rewrite": sim_rewrite,
        "relation_sim_bank": sim_bank,
        "edit_advantage": adv,
        "edit_advantage_source": adv_source,
        "base_alignment_source": alignment_source,
        "prompt_relation_text": prompt_relation,
        "rewrite_relation_text": rewrite_relation,
    }


gate_dataset = []
for spec in METHOD_SPECS:
    normal_rows = load_rows_for_method(spec["normal_run_dir"], spec["method"], source="normal")
    stress_rows = load_rows_for_method(spec["stress_run_dir"], spec["method"], source="stress")
    method_index = {}
    method_index.update(index_rows(normal_rows, "normal"))
    method_index.update(index_rows(stress_rows, "stress"))
    occurrence_counts = defaultdict(int)
    for key, method_row in method_index.items():
        edit_id, bucket, _ = key[:3]
        occurrence_key = (edit_id, bucket)
        occurrence_index = occurrence_counts[occurrence_key]
        occurrence_counts[occurrence_key] += 1
        base_row, alignment_source = find_base_row(method_row, "stress" if bucket in {"same_subject_template", "generation", "attribute"} else "normal", key, occurrence_index)
        if base_row is None:
            continue
        gate_dataset.append(feature_row(base_row, method_row, spec, key, alignment_source))

assert not base_missing_examples, f"Missing base rows after fallback: {base_missing_examples[:10]}"
assert gate_dataset, "No replay dataset rows built"

expected_edit_buckets = ["rewrite", "declarative_paraphrases", "near_locality", "far_locality"]
expected_stress = {
    "same_subject_template": (200, 600),
    "generation": (200, 400),
}
by_method_bucket = defaultdict(list)
for row in gate_dataset:
    by_method_bucket[(row["replay_method_label"], row["bucket"])].append(row)

for spec in METHOD_SPECS:
    label = spec["replay_method_label"]
    for bucket in expected_edit_buckets:
        rows = by_method_bucket[(label, bucket)]
        edit_ids = {row["edit_id"] for row in rows}
        assert len(edit_ids) == 200, f"{label}/{bucket}: expected 200 edits, got {len(edit_ids)}"
    for bucket, (expected_edits, expected_rows) in expected_stress.items():
        rows = by_method_bucket[(label, bucket)]
        edit_ids = {row["edit_id"] for row in rows}
        assert len(edit_ids) == expected_edits, f"{label}/{bucket}: expected {expected_edits} edits, got {len(edit_ids)}"
        assert len(rows) == expected_rows, f"{label}/{bucket}: expected {expected_rows} rows, got {len(rows)}"

ADVANTAGE_FEATURE_AVAILABLE = any(row.get("edit_advantage") is not None for row in gate_dataset)
ADVANTAGE_SOURCES = sorted({row.get("edit_advantage_source") for row in gate_dataset if row.get("edit_advantage_source")})

bucket_counts = Counter(row["bucket"] for row in gate_dataset)
method_counts = Counter(row["replay_method_label"] for row in gate_dataset)
print("gate dataset rows:", len(gate_dataset))
print("bucket counts:", dict(bucket_counts))
print("method row counts:", dict(method_counts))
print("base alignment counts:", dict(base_alignment_counts))
print("valid pre-decode advantage feature available:", ADVANTAGE_FEATURE_AVAILABLE)
print("advantage sources:", ADVANTAGE_SOURCES)

write_csv(OUT_REPORT / "gate_replay_dataset.csv", gate_dataset)
print("Wrote:", OUT_REPORT / "gate_replay_dataset.csv")


## Gate Grid

First eligible gates:

```text
Gate 1: subject + relation_sim_rewrite
Gate 2: subject + relation_sim_bank
Gate 3: subject + (relation_sim_rewrite OR relation_sim_bank)
```

No selectable gate uses `exact_rate`, `target_false_positive_rate`, or any other post-decode outcome as an activation feature.


In [ ]:
REL_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]
BANK_THRESHOLDS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]

GATE_CONFIGS = []
for tau in REL_THRESHOLDS:
    GATE_CONFIGS.append({
        "gate_family": "relation_lexical",
        "gate_id": f"relation_lexical_tau{tau:.2f}",
        "tau_rel": tau,
        "eligible_for_selection": True,
        "oracle_diagnostic": False,
    })
for tau in BANK_THRESHOLDS:
    GATE_CONFIGS.append({
        "gate_family": "relation_bank",
        "gate_id": f"relation_bank_tau{tau:.2f}",
        "tau_bank": tau,
        "eligible_for_selection": True,
        "oracle_diagnostic": False,
    })
for tau_rel in REL_THRESHOLDS:
    for tau_bank in BANK_THRESHOLDS:
        GATE_CONFIGS.append({
            "gate_family": "hybrid_relation_or",
            "gate_id": f"hybrid_or_rel{tau_rel:.2f}_bank{tau_bank:.2f}",
            "tau_rel": tau_rel,
            "tau_bank": tau_bank,
            "eligible_for_selection": True,
            "oracle_diagnostic": False,
        })

print("gate configs:", len(GATE_CONFIGS))
print("by family:", dict(Counter(g["gate_family"] for g in GATE_CONFIGS)))


## Replay And Report Construction


In [ ]:
thresholds = json.load(THRESHOLDS.open())
selfloc_base = float(thresholds["selfloc_base"])
near_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["near_locality"])
far_tfpr_budget = float(thresholds["target_false_positive_rate_budget_by_bucket"]["far_locality"])
malformed_budget = float(thresholds["malformed_span_rate_budget"])
gpu_budget = float(thresholds["gpu_minutes_per_edit_budget"])

MIN_REWRITE_EXACT = 0.25
MIN_DECLARATIVE_PARAPHRASE_EXACT = 0.20
MIN_SELECTION_SCORE = 0.30


def gate_on(row, cfg):
    if not bool(row["subject_match"]):
        return False
    family = cfg["gate_family"]
    if family == "relation_lexical":
        return float(row["relation_sim_rewrite"]) >= float(cfg["tau_rel"])
    if family == "relation_bank":
        return float(row["relation_sim_bank"]) >= float(cfg["tau_bank"])
    if family == "hybrid_relation_or":
        return (
            float(row["relation_sim_rewrite"]) >= float(cfg["tau_rel"])
            or float(row["relation_sim_bank"]) >= float(cfg["tau_bank"])
        )
    raise ValueError(f"Unknown gate family: {family}")


def choose_value(row, gate_active, key):
    prefix = "method" if gate_active else "base"
    return row.get(f"{prefix}_{key}")


def compose_replay_rows(rows, cfg):
    replay_rows = []
    for row in rows:
        active = gate_on(row, cfg)
        replay_label = f"{row['replay_method_label']}__{cfg['gate_id']}"
        replay_rows.append({
            "protocol_version": PROTOCOL_VERSION,
            "split_role": "dev_tune_200",
            "method": row["replay_method_label"],
            "method_variant": replay_label,
            "gate_family": cfg["gate_family"],
            "gate_id": cfg["gate_id"],
            "eligible_for_selection": bool(cfg.get("eligible_for_selection", False)),
            "oracle_diagnostic": bool(cfg.get("oracle_diagnostic", False)),
            "edit_id": row["edit_id"],
            "case_id": row["case_id"],
            "bucket": row["bucket"],
            "prompt_id": row["prompt_id"],
            "target_length_bin": row.get("target_length_bin"),
            "relation_id": row.get("relation_id"),
            "gate_activation_rate": float(active),
            "exact_rate": choose_value(row, active, "exact_rate"),
            "token_f1": choose_value(row, active, "token_f1"),
            "malformed_rate": choose_value(row, active, "malformed_rate"),
            "target_false_positive_rate": choose_value(row, active, "target_false_positive_rate"),
            "sparse_guidance_kl": row.get("method_sparse_guidance_kl") if active else 0.0,
            "subject_match": row["subject_match"],
            "relation_sim_rewrite": row["relation_sim_rewrite"],
            "relation_sim_bank": row["relation_sim_bank"],
            "edit_advantage": row.get("edit_advantage"),
            "edit_advantage_source": row.get("edit_advantage_source"),
            "prompt": row["prompt"],
            "base_output": row["base_output"],
            "method_output": row["method_output"],
            "replay_output": row["method_output"] if active else row["base_output"],
        })
    return replay_rows


def aggregate_replay_rows(rows):
    by_bucket_edit = defaultdict(list)
    for row in rows:
        by_bucket_edit[(row["bucket"], str(row["edit_id"]))].append(row)
    by_bucket = defaultdict(list)
    for (bucket, edit_id), items in by_bucket_edit.items():
        by_bucket[bucket].append({
            "edit_id": edit_id,
            "exact_rate": mean_or_none(item.get("exact_rate") for item in items),
            "token_f1": mean_or_none(item.get("token_f1") for item in items),
            "malformed_rate": mean_or_none(item.get("malformed_rate") for item in items),
            "target_false_positive_rate": mean_or_none(item.get("target_false_positive_rate") for item in items),
            "sparse_guidance_kl": mean_or_none(item.get("sparse_guidance_kl") for item in items),
            "gate_activation_rate": mean_or_none(item.get("gate_activation_rate") for item in items),
            "num_prompt_rows": len(items),
        })
    summary = {}
    for bucket, edit_rows in by_bucket.items():
        summary[bucket] = {
            "num_edits": len(edit_rows),
            "num_prompt_rows": sum(item["num_prompt_rows"] for item in edit_rows),
            "mean_exact_rate": mean_or_none(item.get("exact_rate") for item in edit_rows),
            "mean_token_f1": mean_or_none(item.get("token_f1") for item in edit_rows),
            "mean_malformed_rate": mean_or_none(item.get("malformed_rate") for item in edit_rows),
            "mean_target_false_positive_rate": mean_or_none(item.get("target_false_positive_rate") for item in edit_rows),
            "mean_sparse_guidance_kl": mean_or_none(item.get("sparse_guidance_kl") for item in edit_rows),
            "gate_activation_rate": mean_or_none(item.get("gate_activation_rate") for item in edit_rows),
        }
    return summary


def useful_edit(row):
    return (
        float(row.get("rewrite_exact") or 0.0) >= MIN_REWRITE_EXACT
        and float(row.get("declarative_paraphrases_exact") or 0.0) >= MIN_DECLARATIVE_PARAPHRASE_EXACT
        and float(row.get("selection_score") or 0.0) >= MIN_SELECTION_SCORE
    )


def safe_get(summary, bucket, key, default=None):
    return summary.get(bucket, {}).get(key, default)

# Base stress budgets come from actual Step 3D base rows.
base_all_rows = []
for row in base_normal_rows:
    base_all_rows.append({
        "method_variant": "base_replay",
        "edit_id": str(row.get("edit_id") or row.get("case_id")),
        "case_id": str(row.get("case_id") or row.get("edit_id")),
        "bucket": row_bucket(row, "normal"),
        "prompt_id": row.get("prompt_id") or row.get("prompt_uid"),
        "exact_rate": metric(row, "exact_rate"),
        "token_f1": metric(row, "token_f1"),
        "malformed_rate": metric(row, "malformed_rate"),
        "target_false_positive_rate": metric(row, "target_false_positive_rate"),
        "sparse_guidance_kl": 0.0,
        "gate_activation_rate": 0.0,
    })
for row in base_stress_rows:
    base_all_rows.append({
        "method_variant": "base_replay",
        "edit_id": str(row.get("edit_id") or row.get("case_id")),
        "case_id": str(row.get("case_id") or row.get("edit_id")),
        "bucket": row_bucket(row, "stress"),
        "prompt_id": row.get("prompt_id") or row.get("prompt_uid"),
        "exact_rate": metric(row, "exact_rate"),
        "token_f1": metric(row, "token_f1"),
        "malformed_rate": metric(row, "malformed_rate"),
        "target_false_positive_rate": metric(row, "target_false_positive_rate"),
        "sparse_guidance_kl": 0.0,
        "gate_activation_rate": 0.0,
    })
base_summary = aggregate_replay_rows(base_all_rows)
same_subject_tfpr_budget = (base_summary["same_subject_template"].get("mean_target_false_positive_rate") or 0.0) + 0.03
generation_tfpr_budget = (base_summary["generation"].get("mean_target_false_positive_rate") or 0.0) + 0.03
stress_malformed_budget = max(0.05, (base_summary["same_subject_template"].get("mean_malformed_rate") or 0.0) + 0.02)

rows_by_method = defaultdict(list)
for row in gate_dataset:
    rows_by_method[row["replay_method_label"]].append(row)

method_bucket_rows = []
stress_summary_rows = []
activation_rows = []
grid_rows = []
selection_rows = []
composed_cache = {}

for cfg in GATE_CONFIGS:
    for spec in METHOD_SPECS:
        method_rows = rows_by_method[spec["replay_method_label"]]
        replay_rows = compose_replay_rows(method_rows, cfg)
        replay_label = replay_rows[0]["method_variant"]
        summary = aggregate_replay_rows(replay_rows)
        composed_cache[replay_label] = replay_rows

        rewrite = safe_get(summary, "rewrite", "mean_exact_rate")
        paraphrase = safe_get(summary, "declarative_paraphrases", "mean_exact_rate")
        qa = safe_get(summary, "qa_format_generalization", "mean_exact_rate")
        locality = mean_or_none([
            safe_get(summary, "near_locality", "mean_exact_rate"),
            safe_get(summary, "far_locality", "mean_exact_rate"),
        ])
        clipped_locality = min((locality or 0.0) / max(selfloc_base, 1e-6), 1.0)
        selection_score = harmonic_mean([rewrite or 0.0, paraphrase or 0.0, clipped_locality])
        near_tfpr = safe_get(summary, "near_locality", "mean_target_false_positive_rate")
        far_tfpr = safe_get(summary, "far_locality", "mean_target_false_positive_rate")
        same_subject_tfpr = safe_get(summary, "same_subject_template", "mean_target_false_positive_rate")
        generation_tfpr = safe_get(summary, "generation", "mean_target_false_positive_rate")
        max_malformed = max((bucket.get("mean_malformed_rate") or 0.0) for bucket in summary.values())
        primary_kl = mean_or_none([
            safe_get(summary, "rewrite", "mean_sparse_guidance_kl"),
            safe_get(summary, "declarative_paraphrases", "mean_sparse_guidance_kl"),
        ])
        rewrite_activation = safe_get(summary, "rewrite", "gate_activation_rate")
        paraphrase_activation = safe_get(summary, "declarative_paraphrases", "gate_activation_rate")
        qa_activation = safe_get(summary, "qa_format_generalization", "gate_activation_rate")
        same_subject_activation = safe_get(summary, "same_subject_template", "gate_activation_rate")
        generation_activation = safe_get(summary, "generation", "gate_activation_rate")

        near_tfpr_pass = near_tfpr is not None and near_tfpr <= near_tfpr_budget
        far_tfpr_pass = far_tfpr is not None and far_tfpr <= far_tfpr_budget
        same_subject_tfpr_pass = same_subject_tfpr is not None and same_subject_tfpr <= same_subject_tfpr_budget
        generation_tfpr_pass = generation_tfpr is not None and generation_tfpr <= generation_tfpr_budget
        malformed_pass = max_malformed <= malformed_budget and max_malformed <= stress_malformed_budget
        activation_pass = (rewrite_activation or 0.0) >= 0.95 and (paraphrase_activation or 0.0) >= 0.85
        eligible_for_selection = bool(cfg.get("eligible_for_selection", False))
        oracle_diagnostic = bool(cfg.get("oracle_diagnostic", False))

        row = {
            "replay_label": replay_label,
            "replay_method_label": spec["replay_method_label"],
            "source_label": spec["source_label"],
            "family": spec["family"],
            "guidance_scale": spec["guidance_scale"],
            "gate_family": cfg["gate_family"],
            "gate_id": cfg["gate_id"],
            "eligible_for_selection": eligible_for_selection,
            "oracle_diagnostic": oracle_diagnostic,
            "rewrite_exact": rewrite,
            "declarative_paraphrases_exact": paraphrase,
            "qa_format_generalization_exact": qa,
            "locality_exact": locality,
            "selfloc_base": selfloc_base,
            "clipped_self_normalized_locality": clipped_locality,
            "selection_score": selection_score,
            "primary_sparse_guidance_kl": primary_kl,
            "near_locality_tfpr": near_tfpr,
            "far_locality_tfpr": far_tfpr,
            "same_subject_template_tfpr": same_subject_tfpr,
            "generation_tfpr": generation_tfpr,
            "max_malformed_rate": max_malformed,
            "rewrite_gate_activation": rewrite_activation,
            "declarative_paraphrases_gate_activation": paraphrase_activation,
            "qa_format_generalization_gate_activation": qa_activation,
            "same_subject_template_gate_activation": same_subject_activation,
            "generation_gate_activation": generation_activation,
            "near_tfpr_pass": near_tfpr_pass,
            "far_tfpr_pass": far_tfpr_pass,
            "same_subject_template_tfpr_pass": same_subject_tfpr_pass,
            "generation_tfpr_pass": generation_tfpr_pass,
            "malformed_pass": malformed_pass,
            "activation_pass": activation_pass,
            "template_stress_pass": same_subject_tfpr_pass and malformed_pass and activation_pass,
            "strict_stress_pass": same_subject_tfpr_pass and generation_tfpr_pass and malformed_pass and activation_pass,
        }
        row["edit_useful_pass"] = useful_edit(row)
        preliminary_pass = row["template_stress_pass"] and row["edit_useful_pass"] and near_tfpr_pass and far_tfpr_pass
        row["preliminary_candidate_pass"] = preliminary_pass
        if preliminary_pass and eligible_for_selection:
            row["decision_status"] = "keep_dev_candidate"
        elif preliminary_pass and oracle_diagnostic:
            row["decision_status"] = "oracle_diagnostic_candidate"
        else:
            row["decision_status"] = "reject_or_diagnostic"
        row["constraint_violations"] = ";".join([
            name for name, passed in [
                ("eligible_for_selection", eligible_for_selection),
                ("rewrite_activation", (rewrite_activation or 0.0) >= 0.95),
                ("declarative_activation", (paraphrase_activation or 0.0) >= 0.85),
                ("near_tfpr", near_tfpr_pass),
                ("far_tfpr", far_tfpr_pass),
                ("same_subject_template_tfpr", same_subject_tfpr_pass),
                ("malformed", malformed_pass),
                ("edit_useful", row["edit_useful_pass"]),
            ]
            if not passed
        ])
        grid_rows.append(row)
        selection_rows.append(row)

        for bucket, bucket_summary in sorted(summary.items()):
            bucket_row = {
                "replay_label": replay_label,
                "replay_method_label": spec["replay_method_label"],
                "family": spec["family"],
                "guidance_scale": spec["guidance_scale"],
                "gate_family": cfg["gate_family"],
                "gate_id": cfg["gate_id"],
                "eligible_for_selection": eligible_for_selection,
                "oracle_diagnostic": oracle_diagnostic,
                "bucket": bucket,
                **bucket_summary,
            }
            method_bucket_rows.append(bucket_row)
            activation_rows.append({
                "replay_label": replay_label,
                "replay_method_label": spec["replay_method_label"],
                "family": spec["family"],
                "guidance_scale": spec["guidance_scale"],
                "gate_family": cfg["gate_family"],
                "gate_id": cfg["gate_id"],
                "eligible_for_selection": eligible_for_selection,
                "oracle_diagnostic": oracle_diagnostic,
                "bucket": bucket,
                "gate_activation_rate": bucket_summary.get("gate_activation_rate"),
                "num_edits": bucket_summary.get("num_edits"),
                "num_prompt_rows": bucket_summary.get("num_prompt_rows"),
            })
            if bucket in {"same_subject_template", "generation"}:
                stress_summary_rows.append(bucket_row)

selection_rows = sorted(
    selection_rows,
    key=lambda row: (
        row["decision_status"] != "keep_dev_candidate",
        row["decision_status"] != "oracle_diagnostic_candidate",
        -(float(row.get("selection_score") or 0.0)),
    ),
)
best_gate_candidates = [
    row for row in selection_rows
    if row["decision_status"] == "keep_dev_candidate" and row["eligible_for_selection"]
]
oracle_gate_candidates = [
    row for row in selection_rows
    if row.get("oracle_diagnostic") and row.get("preliminary_candidate_pass")
]

write_csv(OUT_REPORT / "gate_grid_summary.csv", grid_rows)
write_csv(OUT_REPORT / "gate_activation_summary.csv", activation_rows)
write_csv(OUT_REPORT / "gate_replay_method_bucket.csv", method_bucket_rows)
write_csv(OUT_REPORT / "gate_replay_stress_summary.csv", stress_summary_rows)
write_csv(OUT_REPORT / "gate_replay_selection.csv", selection_rows)
write_csv(OUT_REPORT / "best_gate_candidates.csv", best_gate_candidates)
write_csv(OUT_REPORT / "oracle_gate_candidates.csv", oracle_gate_candidates)

print("grid rows:", len(grid_rows))
print("eligible kept candidates:", len(best_gate_candidates))
print("oracle diagnostic candidates:", len(oracle_gate_candidates))
print("Top rows:")
for row in selection_rows[:10]:
    print(row["decision_status"], row["replay_label"], row["selection_score"], row["constraint_violations"])


## Paired Bootstrap And Output Samples

Bootstrap is limited to the best replay candidates so this remains a cheap report step. If no candidate survives, the bootstrap file records the best diagnostic rows by score.


In [ ]:
# Build a small base replay row set for bootstrap.
base_bootstrap_rows = [
    {**row, "method_variant": "base_replay"}
    for row in base_all_rows
]

eligible_selection_rows = [row for row in selection_rows if row.get("eligible_for_selection")]
bootstrap_targets = best_gate_candidates[:12]
if not bootstrap_targets:
    bootstrap_targets = eligible_selection_rows[:12]

bootstrap_rows = []
for candidate in bootstrap_targets:
    label = candidate["replay_label"]
    pair_rows = composed_cache[label] + base_bootstrap_rows
    for bucket, metric_name, row_metric in [
        ("rewrite", "rewrite_exact", "exact_rate"),
        ("declarative_paraphrases", "declarative_paraphrases_exact", "exact_rate"),
        ("near_locality", "near_tfpr", "target_false_positive_rate"),
        ("far_locality", "far_tfpr", "target_false_positive_rate"),
        ("same_subject_template", "same_subject_template_tfpr", "target_false_positive_rate"),
        ("generation", "generation_tfpr", "target_false_positive_rate"),
        ("same_subject_template", "same_subject_template_malformed", "malformed_rate"),
    ]:
        boot = paired_bootstrap_delta_by_case(
            pair_rows,
            candidate_method=label,
            baseline_method="base_replay",
            bucket=bucket,
            metric=row_metric,
            samples=1000,
            seed=0,
        )
        if boot:
            bootstrap_rows.append({
                "candidate": label,
                "baseline": "base_replay",
                "bucket": bucket,
                "metric": metric_name,
                "mean_delta_candidate_minus_baseline": boot["mean_delta"],
                "ci_lower": boot["ci_low"],
                "ci_upper": boot["ci_high"],
                "num_edits": boot["num_edits"],
            })

# A few head-to-head comparisons among top surviving/diagnostic eligible rows.
head_to_head = []
for left in bootstrap_targets[:8]:
    for right in bootstrap_targets[:8]:
        if left["replay_label"] == right["replay_label"]:
            continue
        if left["family"] == right["family"]:
            continue
        pair = tuple(sorted([left["replay_label"], right["replay_label"]]))
        if pair in head_to_head:
            continue
        head_to_head.append(pair)
        left_label, right_label = left["replay_label"], right["replay_label"]
        pair_rows = composed_cache[left_label] + composed_cache[right_label]
        for bucket, metric_name, row_metric in [
            ("rewrite", "rewrite_exact", "exact_rate"),
            ("declarative_paraphrases", "declarative_paraphrases_exact", "exact_rate"),
            ("same_subject_template", "same_subject_template_tfpr", "target_false_positive_rate"),
        ]:
            boot = paired_bootstrap_delta_by_case(
                pair_rows,
                candidate_method=left_label,
                baseline_method=right_label,
                bucket=bucket,
                metric=row_metric,
                samples=1000,
                seed=1,
            )
            if boot:
                bootstrap_rows.append({
                    "candidate": left_label,
                    "baseline": right_label,
                    "bucket": bucket,
                    "metric": metric_name,
                    "mean_delta_candidate_minus_baseline": boot["mean_delta"],
                    "ci_lower": boot["ci_low"],
                    "ci_upper": boot["ci_high"],
                    "num_edits": boot["num_edits"],
                })

write_csv(OUT_REPORT / "gate_replay_paired_bootstrap.csv", bootstrap_rows)

sample_rows = []
for candidate in bootstrap_targets[:8]:
    label = candidate["replay_label"]
    rows = composed_cache[label]
    groups = defaultdict(list)
    for row in rows:
        if row["bucket"] == "same_subject_template" and (row.get("target_false_positive_rate") or 0.0) > 0.0:
            groups["same_subject_leakage"].append(row)
        if row["bucket"] == "same_subject_template" and row.get("gate_activation_rate") == 0.0:
            groups["same_subject_gate_off"].append(row)
        if row["bucket"] in {"rewrite", "declarative_paraphrases"} and row.get("gate_activation_rate") == 0.0:
            groups["edit_prompt_gate_off"].append(row)
        if row["bucket"] in {"rewrite", "declarative_paraphrases"} and row.get("gate_activation_rate") == 1.0:
            groups["edit_prompt_gate_on"].append(row)
        if (row.get("malformed_rate") or 0.0) > 0.0:
            groups["malformed"].append(row)
    for category, items in groups.items():
        for row in items[:10]:
            sample_rows.append({
                "sample_category": category,
                "replay_label": label,
                "gate_family": row["gate_family"],
                "gate_id": row["gate_id"],
                "eligible_for_selection": row.get("eligible_for_selection"),
                "oracle_diagnostic": row.get("oracle_diagnostic"),
                "edit_id": row["edit_id"],
                "case_id": row["case_id"],
                "relation_id": row.get("relation_id"),
                "target_length_bin": row.get("target_length_bin"),
                "bucket": row["bucket"],
                "prompt_id": row.get("prompt_id"),
                "prompt": row.get("prompt"),
                "subject_match": row.get("subject_match"),
                "relation_sim_rewrite": row.get("relation_sim_rewrite"),
                "relation_sim_bank": row.get("relation_sim_bank"),
                "edit_advantage": row.get("edit_advantage"),
                "edit_advantage_source": row.get("edit_advantage_source"),
                "gate_activation_rate": row.get("gate_activation_rate"),
                "target_false_positive_rate": row.get("target_false_positive_rate"),
                "exact_rate": row.get("exact_rate"),
                "malformed_rate": row.get("malformed_rate"),
                "base_output": row.get("base_output"),
                "method_output": row.get("method_output"),
                "replay_output": row.get("replay_output"),
            })
write_csv(OUT_REPORT / "gate_replay_output_samples.csv", sample_rows)
print("bootstrap rows:", len(bootstrap_rows))
print("sample rows:", len(sample_rows))


## Optional Plot And Report Summary


In [ ]:
try:
    import matplotlib.pyplot as plt

    plot_rows = [
        row for row in selection_rows
        if row.get("same_subject_template_tfpr") is not None and row.get("selection_score") is not None and row.get("eligible_for_selection")
    ]
    colors = {
        "prompt_memory": "#1f77b4",
        "myopic_score": "#ff7f0e",
        "mc_bridge": "#d62728",
        "no_rollout_bridge": "#2ca02c",
    }
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for row in plot_rows:
        ax.scatter(
            float(row["same_subject_template_tfpr"]),
            float(row["selection_score"]),
            color=colors.get(row["family"], "#444444"),
            alpha=0.45 if row["decision_status"] != "keep_dev_candidate" else 0.95,
            s=18 if row["decision_status"] != "keep_dev_candidate" else 42,
        )
    ax.axvline(same_subject_tfpr_budget, color="#555555", linestyle="--", linewidth=1)
    ax.set_xlabel("Same-subject template TFPR")
    ax.set_ylabel("Replay selection score")
    ax.set_title("Hybrid Gate Replay: Eligible Gates Only")
    fig.tight_layout()
    fig.savefig(OUT_REPORT / "gate_replay_tradeoff.png", dpi=200)
    plt.close(fig)
    print("Wrote plot:", OUT_REPORT / "gate_replay_tradeoff.png")
except Exception as exc:
    print("Plot skipped:", repr(exc))

ELIGIBLE_GATE_FAMILIES = ["relation_lexical", "relation_bank", "hybrid_relation_or"]
ORACLE_GATE_FAMILIES = []

report = {
    "protocol_version": PROTOCOL_VERSION,
    "split_role": "dev_tune_200",
    "analysis_500_used": False,
    "final_test_used": False,
    "stage": "Step 3E.0 gate-only replay",
    "gate_families": ELIGIBLE_GATE_FAMILIES,
    "eligible_gate_families": ELIGIBLE_GATE_FAMILIES,
    "oracle_gate_families": ORACLE_GATE_FAMILIES,
    "uses_outcome_derived_advantage": False,
    "advantage_feature_available": ADVANTAGE_FEATURE_AVAILABLE,
    "advantage_feature_sources": ADVANTAGE_SOURCES,
    "advantage_gates_eligible_for_selection": False,
    "primary_stress_bucket": "same_subject_template",
    "qa_format_generalization": "secondary_diagnostic",
    "source_outputs": "Existing subject-gated and base outputs; no new LLaDA decoding.",
    "no_paraphrase_template_leakage": True,
    "relation_bank_source": "dev_tune_200 rewrite templates only",
    "selfloc_base": selfloc_base,
    "budgets": {
        "near_locality_tfpr": near_tfpr_budget,
        "far_locality_tfpr": far_tfpr_budget,
        "same_subject_template_tfpr": same_subject_tfpr_budget,
        "generation_tfpr": generation_tfpr_budget,
        "malformed_rate": malformed_budget,
        "stress_malformed_rate": stress_malformed_budget,
        "gpu_minutes_per_edit": gpu_budget,
    },
    "edit_useful_thresholds": {
        "min_rewrite_exact": MIN_REWRITE_EXACT,
        "min_declarative_paraphrases_exact": MIN_DECLARATIVE_PARAPHRASE_EXACT,
        "min_selection_score": MIN_SELECTION_SCORE,
    },
    "acceptance_criteria": {
        "rewrite_activation_min": 0.95,
        "declarative_paraphrase_activation_min": 0.85,
        "same_subject_template_tfpr_max": "base + 0.03",
        "near_far_tfpr_budgets": True,
        "malformed_rate_max": 0.05,
        "edit_useful_pass": True,
        "eligible_for_selection": True,
    },
    "candidate_source_labels": [spec["source_label"] for spec in METHOD_SPECS],
    "candidate_replay_method_labels": [spec["replay_method_label"] for spec in METHOD_SPECS],
    "base_alignment_counts": dict(base_alignment_counts),
    "base_alignment_policy": "Exact prompt id, prompt hash, normalized prompt hash, occurrence-order, then inactive-gate fallback only when the original subject gate was off.",
    "num_gate_configs": len(GATE_CONFIGS),
    "num_grid_rows": len(grid_rows),
    "num_keep_dev_candidates": len(best_gate_candidates),
    "num_oracle_gate_candidates": len(oracle_gate_candidates),
    "artifacts": {
        "gate_replay_dataset": str(OUT_REPORT / "gate_replay_dataset.csv"),
        "gate_grid_summary": str(OUT_REPORT / "gate_grid_summary.csv"),
        "gate_activation_summary": str(OUT_REPORT / "gate_activation_summary.csv"),
        "gate_replay_method_bucket": str(OUT_REPORT / "gate_replay_method_bucket.csv"),
        "gate_replay_stress_summary": str(OUT_REPORT / "gate_replay_stress_summary.csv"),
        "gate_replay_selection": str(OUT_REPORT / "gate_replay_selection.csv"),
        "gate_replay_paired_bootstrap": str(OUT_REPORT / "gate_replay_paired_bootstrap.csv"),
        "gate_replay_output_samples": str(OUT_REPORT / "gate_replay_output_samples.csv"),
        "best_gate_candidates": str(OUT_REPORT / "best_gate_candidates.csv"),
        "oracle_gate_candidates": str(OUT_REPORT / "oracle_gate_candidates.csv"),
        "plot": str(OUT_REPORT / "gate_replay_tradeoff.png"),
    },
}
with (OUT_REPORT / "report_summary.json").open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Wrote report summary:", OUT_REPORT / "report_summary.json")
print(json.dumps({
    "base_alignment_counts": dict(base_alignment_counts),
    "base_alignment_policy": "Exact prompt id, prompt hash, normalized prompt hash, occurrence-order, then inactive-gate fallback only when the original subject gate was off.",
    "num_gate_configs": len(GATE_CONFIGS),
    "num_grid_rows": len(grid_rows),
    "num_keep_dev_candidates": len(best_gate_candidates),
    "num_oracle_gate_candidates": len(oracle_gate_candidates),
}, indent=2))


## Expected Decision

Inspect first:

```text
best_gate_candidates.csv
oracle_gate_candidates.csv
gate_replay_selection.csv
gate_replay_stress_summary.csv
gate_activation_summary.csv
gate_replay_output_samples.csv
```

If no replay gate satisfies:

```text
rewrite activation >= 0.95
declarative paraphrase activation >= 0.85
same_subject_template TFPR <= base + 0.03
near/far TFPR budgets pass
malformed_rate <= 0.05
edit_useful_pass = true
```

then do not spend GPU on Step 3E.1. The subject-family runtime gate should be treated as rejected and the next scientific move is Direction 2.

If one or more gates pass, choose at most the top two gates and top three methods for actual dev-only decoding in Step 3E.1.

Do not move to `analysis_500` after this notebook.
